<img src="https://github.com/moroneyt/MXB301/raw/main/resources/qutlogo.jpg">

# MXB301 Mathematics of AI
# Lesson 5: Optimisation

#### Tim Moroney, 2026


A lesson where we cover various numerical optimisation methods and finally train our character model with real data.

# Package management

In [ ]:
import Pkg
if haskey(ENV, "COLAB_GPU") # check if we're on Colab
  if !isfile("/content/MXB301_2026_01_CPU.tgz") # check if we've already downloaded
    # download precompiled Julia environment for Colab
    run(`gdown https://drive.google.com/uc\?id=1mT9XFadzdfK8CWb5a7BYLUkd2RTi2eZc`)

    # replace Colab's Julia environment with downloaded version
    run(`rm -rf /root/.julia`)
    run(`tar -xzf MXB301_2026_01_CPU.tgz -C /root`)
  end
else
  # For any other machine we install the packages in the usual way
  Pkg.activate(".")
  Pkg.add(["CairoMakie", "CodecZlib", "ColorSchemes", "ComponentArrays", "CondaPkg",
           "DifferentiationInterface", "Distributions", "Downloads", "FiniteDiff", "ForwardDiff",
           "HTTP", "JLD2", "LaTeXStrings", "LinearAlgebra", "Lux", "MKL", "MLUtils", "NNlib",
           "NLSolversBase", "OneHotArrays", "Optim", "PythonCall", "QuadGK", "Random",
           "SpecialFunctions", "Statistics", "StatsBase", "ToeplitzMatrices", "Zygote"])
end

using CairoMakie
using DifferentiationInterface
using LaTeXStrings
using LinearAlgebra
using Lux
using Random
using SpecialFunctions
using Statistics

using ComponentArrays: ComponentVector
using Distributions: Normal, Exponential
using Downloads: download
using ForwardDiff: Dual, partials
using JLD2: jldopen
using MLUtils: DataLoader, rand_like, randn_like
using NLSolversBase: only_fg
using NNlib: softmax, sigmoid, scatter as scattergrad, conv, ∇conv_filter, ∇conv_data, DenseConvDims
using OneHotArrays: onehot, onehotbatch, onecold
using StatsBase: crossentropy, sample, Weights
using ToeplitzMatrices: Toeplitz, Hankel
using QuadGK: quadgk

import CodecZlib
import ColorSchemes
import FiniteDiff
import HTTP
import MKL
import Optim
import Zygote

# Set the random seed for reproducibility
rng = Random.seed!(0)

# SVG format scales properly in web pages and PDFs
CairoMakie.activate!(type = "svg")

# Model code

OK let's get straight into it.  Here is our fully-functional, batch-aware model code.

## Setup

In [ ]:
chars = ['a':'z'; ' ']      # we only deal with lowercase text and space
vocab_size = length(chars)  # number of characters in our "vocabulary"

# Mappings between characters and indices (a = 1, b = 2, etc.)
idx_to_char(i) = chars[i]
char_to_idx(c) = findfirst(isequal(c), chars)
string_to_idxs(s) = [char_to_idx(c) for c in s]

## Forward pass

In [ ]:
# Predict probabilities of the next character
function forward(U, p)
    𝕏 = p.We[:, U]                  # 1. embedding
    V = reshape(𝕏, :, size(U,2))    # 2. flatten
    Z1 = p.W1 * V .+ p.b1           # 3. first dense layer
    H1 = tanh.(Z1)                  # 4. activation
    Z2 = p.W2 * H1 .+ p.b2          # 5. second dense layer
    Ŷ = softmax(Z2)                 # 6. softmax for probabilities

    # Return the output along with a cache of intermediate values
    cache = (; U, 𝕏, V, Z1, H1, Z2)
    return Ŷ, cache
end

## Backward pass

In [ ]:
# Backward pass: compute gradient
function backward(p, Y, Ŷ; cache)

    # gradient vector to fill in
    g = zero(p)

    # unpack the cache of intermediate values from the forward pass
    (; U, 𝕏, V, Z1, H1, Z2) = cache

    # Calculate the gradient of the loss with respect to the model parameters.
    # Each rule is simple enough, but take care!
    ∇Z2 = Ŷ - Y                         # 6. softmax with crossentropy loss
    g.W2 = ∇Z2 * H1'                    # 5. second dense layer (weights)
    g.b2 = sum(∇Z2; dims=2)             # 5. second dense layer (bias)
    ∇H1 = p.W2' * ∇Z2                   # 5. second dense layer (data)
    ∇Z1 = (1 .- H1.^2) .* ∇H1           # 4. activation
    g.W1 = ∇Z1 * V'                     # 3. first dense layer (weights)
    g.b1 = sum(∇Z1; dims=2)             # 3. first dense layer (bias)
    ∇V = p.W1' * ∇Z1                    # 3. first dense layer (data)
    ∇𝕏 = reshape(∇V, size(𝕏, 1), :)     # 2. (un)flatten
    g.We = scattergrad(+, ∇𝕏, vec(U))   # 1. embedding (weights)

    return g
end

# Training data
Today we're going to actually train the model!  For this, we need a big set of training data.  As previously mentioned, the pre-trained parameters we've been using in past lessons came from training on the text _Alice's Adventures in Wonderland_.  So let's keep this as our training text.

## Download text
We can download the full text here.

In [ ]:
# Download some free text: we have chosen "Alice's Adventures in Wonderland"
texturl = "http://www.gutenberg.org/files/11/11-0.txt"
response = HTTP.get(texturl)
sourcetext = replace(String(response.body), r"\s+" => " ") # strip excess whitespace

## Preprocess
First we filter the text in lower case, keeping only the letters and spaces,

In [ ]:
# Preprocess text
filteredtext = filter(in(chars), lowercase(sourcetext))  # keep only letters and spaces

##
and then convert to indices.

In [ ]:
traindata = string_to_idxs(filteredtext)      # convert text to indices

# Model hyperparameters

Now we have to decide on the model **hyperparameters**: how big the context size ($C$) will be, the size of the embedding dimension ($d_e$), the size of the hidden dimension.  We will choose these hyperparameters to match the values we are familiar with from past lessons.  (They're called _hyper_ parameters because unlike the weights and biases, they are not learned during training -- they're set once and for all at the outset.)

In code, we have `context_size`$=C$ , `embedding_size`$=d_e$ and `vocab_size`$=|\mathcal{V}|$.

In [ ]:
# Define model hyperparameters -- these are the familiar values we have been using
context_size = 10
embedding_size = 2
hidden_size = 16
input_size = output_size = vocab_size

# Training set

To generate a set of training data, we will simply choose many random sequences of length $C+1$ from the full text of the novel.

## Random sampler
Here's a function that randomly chooses one such sequence of text (which we will then call many times).  The first $C$ characters are returned as $u$, and the $(C+1)$th character is returned as $y$ (converted to one-hot encoding).

In [ ]:
# Randomly sample a sequence of text of a given length, along with the next character
function get_text_sequence(traindata, context_size)

    # A random subset of the full text of appropriate length
    start_idx = rand(1:length(traindata)-context_size)
    end_idx = start_idx + context_size

    u = traindata[start_idx:end_idx-1] # the sequence of character indices
    c = traindata[end_idx]             # the next character after the sequence
    y = onehot(c, 1:vocab_size)        # next character in one-hot encoding

    return u, y
end

## Generate training set
Let's choose, say, $B = 1000$ random samples to be our training set.

In [ ]:
# Get a random training set of u (text sequences) and y (next characters)
B = 1000

UY = [get_text_sequence(traindata, context_size) for n = 1:B] # as (u,y) pairs

##
That's the full training set as an array of $(u,y)$ pairs.  But we want instead a matrix $U$ of all the $u$ inputs, and a separate matrix $Y$ of all the one-hot encodings $y$.  So we need to _stack_ the two separately.

In [ ]:
U = stack(first.(UY))

##

In [ ]:
Y = stack(last.(UY))

# Initial parameter values

Now we are faced with the question of what initial values to use for the parameters before they are trained.  Your first idea might be to initialise all the parameters to zero.  But this would be a bad idea.

The reason is that if all the weights and biases are initialised to zero, every neuron in a layer receives the same input, produces the same output, and therefore has identical gradients during backpropagation.  In which case, they will remain identical forever -- there is no way for gradient descent to split them apart if they all receive the same updates.  Hence we would effectively have only one neuron per layer (albeit duplicated many times), defeating the point of a neural net entirely.

Instead, we need to use _random_ initial weights.  But random according to what distribution?  The following analysis suggests a way to decide.

We consider the propagation of data through forward and backward passes of a dense layer. As we know well by now, these take the form

$$
z = Wh + b \ \textrm{(forward)}\qquad\textrm{and}\qquad \nabla_hL = W^\top \ \nabla_h L \ \textrm{(backward)}
$$

where the weight matrix $W \in \mathbb{R}^{m \times n}$.

The basic idea is to try to keep the _variance_ of the input and output of each layer unchanged -- for both the forward and backward pass.  This should ensure that neither the values nor gradients "explode" (variance gets too large) or "vanish" (variance gets too small) as the data flows through the layers.  Thus, we hope to avoid any instability or stagnation issues during training.

We assume the weights are initialised with zero mean, $\mathbb{E}[W_{ij}] = 0$, and some yet-to-be-determined variance $\textrm{var}[W_{ij}] = \sigma^2$.  Furthermore the components of $h$ and $\nabla_h L$, i.e. the inputs in the forward and backward pass respectively, are assumed independent and identically distributed with mean zero, and independent of the weights.  So it's a very idealised setup, but all we're looking for is a heuristic for initialising the weights, so it's not unreasonable.

Then for the forward pass,
$$
\begin{align*}
z_i &= \sum_{j=1}^n W_{ij}\, h_j + b_i \\
\textrm{var}[z_i] &= \sum_{j=1}^n \textrm{var}[W_{ij}\, h_j] \\
&= n \sigma^2 \textrm{var}[h_i]
\end{align*}
$$
under our assumptions of zero mean and independence.  (We are not considering $b_i$ to be a random variable here, so it contributes nothing to $\textrm{var}[z_i]$).

Therefore for the variance of the input and output of the layer to be equal for the forward pass, we would choose
$$
n\sigma^2 = 1 \implies \sigma^2 = \frac{1}{n}\,.
$$

For the backward pass,
$$
\begin{align*}
(\nabla_h L)_i &= \sum_{j=1}^m W_{ji}\, (\nabla_z L)_j\\
\textrm{var}[(\nabla_h L)_i] &= \sum_{j=1}^m \textrm{var}[W_{ji}\, (\nabla_z L)_j] \\
&= m \sigma^2 \textrm{var}[(\nabla_z L)_i]
\end{align*}
$$
under our assumptions of zero-mean and independence.

Therefore for the variance of the input and output of the layer to be equal for the backward pass, we would choose
$$
m\sigma^2 = 1 \implies \sigma^2 = \frac{1}{m}\,.
$$

Unless $m = n$ we cannot satisfy both requirements, so the natural compromise is to choose $\sigma^2$ as an average of the two expressions
$$
\sigma^2 = \frac{1}{m}\quad\textrm{and}\quad \sigma^2 = \frac{1}{n}\,.
$$
The so-called **Glorot initialisation** uses the _harmonic mean_ of the two:
$$
\frac{1}{\sigma^2} = \frac{1}{2}(m + n)
$$
i.e.
$$
\sigma^2 = \frac{2}{m + n}\,.
$$

So if we are choosing normally-distributed initial weights (which we might as well), this means
$$
W_{ij} \sim \mathcal{N}\left(0, \frac{2}{m + n}\right) \qquad\textrm{(dense layer)}.
$$

As for the _biases_ $b_i$, these can be safely initialised to zero. The random weights $W_{ij}$ already ensure we have enough initial variability for the training process to work with.

Now actually, to be fully rigorous with this analysis, we should have also considered the nonlinear activation function that sits between dense layers.  But fortunately for our choice of activation $\tanh$ we can fall back on the approximation
$$
\tanh(t) = t  - t^3/3 + \ldots
$$
and so for small enough $t$, $\tanh(t) \approx t$ and we are at least somewhat justified in ignoring this contribution.

## Initialisation for dense layers
So we can now confidently initialise our dense layer weight matrices and bias vectors.

In [ ]:
# Helper functions for weight initialisation
glorot_std(m, n) = sqrt(2 / (m + n))
winit_dense(m, n) = glorot_std(m, n) .* randn(m, n)

# Biases are zero
b1 = zeros(hidden_size)
b2 = zeros(output_size)

# Dense layer initial weights
W1 = winit_dense(hidden_size, embedding_size * context_size)
W2 = winit_dense(output_size, hidden_size)

## Embedding layer

What about the embedding layer?  The analysis for dense layers doesn't apply here, since the embedding layer is just a lookup table.  Rather, for an input $u,$ its columns determine the character embeddings $X = W_e[:, u] \in \mathbb{R}^{d_e \times C}$, which in turn becomes the input $v = \textrm{vec}(X) \in \mathbb{R}^{d_eC}$ to the first dense layer.  So we'll opt to initialise the entries in $W_e$ so that this first input vector $v$ has a typical norm of unity; i.e., the entries of $W_e$ have variance $1/(d_eC)$.

In [ ]:
# Embedding layer parameters
We = 1/sqrt(embedding_size * context_size) * randn(embedding_size, input_size)

## Full set of trainable parameters

We now have a well-justified initial set of trainable parameters.

In [ ]:
# The full vector of trainable parameters
p0 = ComponentVector(; We, W1, b1, W2, b2)

# Loss and gradient

Here's the combined loss and gradient function from last lesson.  If ever we only need to compute the loss and no gradient, we will just pass in the option `grad=false`, which will signal the code not to run the backward pass.

In [ ]:
function loss_and_gradient(p; U, Y, grad=true)

    B = size(U, 2) # batch size
    @assert B == size(Y, 2)

    # Forward
    Ŷ, cache = forward(U, p)
    loss = crossentropy(Y, Ŷ) / B
    if !grad return loss end

    # Backward
    g = backward(p, Y, Ŷ; cache) / B

    return loss, g

end

# Gradient descent revisited
[Kochenderfer and Wheeler Section 5.1]

In the last lesson we motivated the idea of _gradient descent_.  In this worksheet we will take the idea a bit more seriously.  First, let's agree to use the _normalised_ negative gradient as the step direction:

$$
d = \frac{-g}{\|g\|} = \frac{-∇_p L(p)}{\|∇_p L(p)\|}\,.
$$

This way, the gradient descent update is

$$
p \leftarrow p + \alpha \, d
$$

and the _learning rate_ $\alpha$ truly is the size of the step taken (because $d$ has unit norm).

As a first attempt, we will try a fixed value of, say, $\alpha = 0.01$.  Here it is now, running for 500 iterations on our training set of 1000 samples.  At last, we're really training our model!  How low can we reduce the loss to?



In [ ]:
# Gradient descent with fixed learning rate
maxiters = 500
p = p0
gd_loss_trace = []  # we'll record the loss at every iteration
α = 0.01  # fixed learning rate

@time for i = 1:maxiters

    loss, g = loss_and_gradient(p; U, Y)
    push!(gd_loss_trace, loss)  # record the loss value

    # descent direction is negative normalised gradient
    d = -normalize(g)

    # take the step
    p += α * d

end

fig, ax = lines(gd_loss_trace/gd_loss_trace[1], label="Gradient desc (fixed α)",
                axis=(title = "Training progress", xlabel = "iterations", ylabel = "normalised loss",
                yscale = log10))

It works, sort of.  But it's terribly inefficient.  A fixed value of the learning rate $\alpha$ is not flexible at all, and would rarely be satisfactory in practice. Better would be to allow $\alpha$ to vary with each step, and ideally somehow choose the optimal, or near-optimal, value each time.

Actually this last idea is somewhat viable.  For given $p$, once you have selected the search direction $d$, the question of what learning rate to use is a _one-dimensional_ optimisation problem.  Define

$$
\phi(\alpha) = L(p + \alpha d)
$$

which is a one-variable function of $\alpha$ alone.  The optimal value of $\alpha$ to use for this step of gradient descent is then

$$
\alpha_\textrm{opt} = \textrm{argmin}_{\alpha \in \mathbb{R}^+} \ \phi(\alpha)\,.
$$

By sampling $\phi(\alpha)$ for different values of $\alpha$, you could build a polynomial model (say) that approximates the loss function along the particular search direction $d$, and use its minimum as the choice of $\alpha$ for that step.  This would come at the additional cost of however many further evaluations of $\phi$ (and hence $L$) you used to build the polynomial model.

This is the concept of **linesearching**, and it's a common feature in optimisation algorithms.  There are many different linesearch algorithms which seek to balance the cost of building the one-dimensional model versus the improvement in convergence obtained through a better choice of $\alpha$.

Let's see just how good this could be, by choosing the near-optimal value of $\alpha$ for each step in our gradient descent.  To do so we will just sample $\phi(\alpha)$ on a fine grid of $\alpha$ values, which is extremely crude and inefficient, but demonstrates the point and ensures we are giving gradient descent the best opportunity to shine.

Here it is now, running for 500 iterations once more.  How much of an improvement will adaptively choosing the learning rate provide?



In [ ]:
# Gradient descent with near-optimal learning rate
maxiters = 500
p = p0
gd2_loss_trace = []      # we'll record the loss at every iteration
αgrid = 1.5.^(-30:0.1:2) # geometric grid of α for linesearch

@time for i = 1:maxiters

    loss, g = loss_and_gradient(p; U, Y)
    push!(gd2_loss_trace, loss)  # record the loss value

    # descent direction is negative normalised gradient
    d = -normalize(g)

    # sample φ on a grid of α to estimate the optimal loss in this direction
    lossgrid = [loss_and_gradient(p + α*d; U, Y, grad=false) for α in αgrid]
    idx = argmin(lossgrid)     # where is the minimum value
    αopt = αgrid[idx]          # optimal learning rate for this step

    # take the optimal step
    p += αopt * d

end

lines!(gd2_loss_trace/gd2_loss_trace[1], label="Gradient desc (optimal)")
axislegend(ax)
fig

## The dreaded zig-zag

So, choosing the optimal value of $\alpha$ can make a big difference to the convergence rate.  But even so, there is still a significant issue that limits the effectiveness of gradient descent.

The optimal value $\alpha_\textrm{opt}$ is found by minimising $$
\phi(\alpha) = L(p + \alpha d)\,.
$$

Differentiating (exercises!), we find
$$
\phi'(\alpha) = \nabla_p L(p + \alpha\, d)^\top d
$$
and so $\alpha_\textrm{opt}$ satisfies
$$
\phi'(\alpha_\textrm{opt}) = \nabla_p L(p + \alpha_\textrm{opt}\, d)^\top d = 0\,.\qquad(*)
$$

But notice what this implies for the the next iteration.  Our updated parameter values would be
$$
p^{(\textrm{new})} = p + \alpha_\textrm{opt}\, d
$$

and we would compute the next search direction for this updated set of parameters $p^{(\textrm{new})}$:
$$
d^{(\textrm{new})} = \frac{-g^{(\textrm{new})}}{\|g^{(\textrm{new})}\|} = \frac{-∇_p L(p^{(\textrm{new})})}{\|∇_p\, L(p^{(\textrm{new})})\|} = \frac{-∇_p\, L(p+\alpha_\textrm{opt} \, d)}{\|∇_p L(p+\alpha_\textrm{opt} \, d)\|}\,.
$$

Now take the dot product with $d$, and use $(*)$ to derive the key result
$$
{d^{(\textrm{new})}}^\top d = 0\,.
$$

That is, in gradient descent, taking the optimal step implies that the new search direction is _orthogonal_ to the previous search direction. This can lead to the infamous "zig-zag" problem of gradient descent, where it takes many alternating steps this way and then that, to traverse narrow valleys in the loss function: see Figure 5.1 in Kochenderfer and Wheeler.

# Newton's method revisted

Recall from last lesson that Newton's method replaces the objective function for the current set of parameters $p$ with its local quadratic approximation

$$
m(s) = a + g^\top s + \frac{1}{2} s^\top H s\,,
$$

satisfying
$$
L(p+\Delta p) \approx m(\Delta p)
$$

where $a = L(p)$ is the loss value, $g = \nabla_p L$ is the gradient, and $H$ is the _Hessian matrix_ of mixed second order partial derivatives
$$
H_{ij} = \frac{\partial^2 L}{\partial p_i\, \partial p_j}\,.
$$

Previously we used calculus to jump straight to the true minimiser of the quadratic function $m$, namely

$$
\Delta p = -H^{-1} g
$$

and used that to propose the update
$$
p^{(\textrm{new})} = p + \Delta p = p - H^{-1} g\,.
$$

However we noted two weaknesses of this approach.  One is that the Hessian matrix is not guaranteed to be positive definite -- far from the actual minimum the local quadratic model $m$ might actually be a saddle.  We overcame this by replacing the Hessian with a regularised version
$$
H_\gamma = H + \gamma I
$$
where $\gamma$ is a positive factor.  A choice like $\gamma = c\|g\|$ can keep things well behaved when starting the iterations, while ensuring we recover the true Newton update in the limit as we converge.

The second weakness was the need to solve the (now regularised) linear system
$$
H_\gamma \Delta p = -g
$$
which is a dense $P \times P$ linear system.

Now we will seek to address this expense, by not actually solving the linear system!  Instead, we will treat the local quadratic model $m$ as an objective function in its own right, and perform a "gradient descent" on _it_, as an iterative approach to finding its minimum.

# Conjugate gradient
[Kochenderfer and Wheeler Section 5.2]

Because of the relative simplicity of the local quadratic model
$$
m(s) = a + g^\top s + \frac{1}{2} s^\top H_\gamma s
$$
we can make several improvements on vanilla gradient descent.

First, we can solve analytically for the optimal learning rate.  Along a given search direction $d$ we have
$$
\phi(\alpha) = m(s + \alpha d)
$$
and so (exercises!)
$$
{\phi}'(\alpha) = d^\top (g + H_\gamma s) + \alpha\, d^\top H_\gamma d\,.
$$
Hence $\alpha_\textrm{opt}$ satisfies
$$
\phi'(\alpha_\textrm{opt}) = d^\top (g + H_\gamma s) + \alpha_\textrm{opt}\, d^\top H_\gamma d = 0\\
$$
i.e.
$$
\alpha_\textrm{opt} = \frac{-d^\top(g + H_\gamma s)}{d^\top H_\gamma d} = \frac{-d^\top r}{d^\top H_\gamma d}
$$
where we have let
$$
r = \nabla_s m = g + H_\gamma s \,.
$$

So stepping this optimal distance along the direction $d$, we obtain the next iterate
$$
s^{(\textrm{new})} = s + \alpha_\textrm{opt}\, d
$$
and
$$
r^{(\textrm{new})} = g + H_\gamma  s^{(\textrm{new})} = g + H_\gamma (s + \alpha_\textrm{opt}\, d) = r + \alpha_\textrm{opt}\, H_\gamma d \,.
$$

The second improvement over vanilla gradient descent is to alter the formula for the search directions.  Instead of just choosing the negative gradient $d = -\nabla_s m = -r$, we choose $d$ as a weighted combination of the negative gradient and the previous search direction:
$$
d^{(\textrm{new})} = -r^{(\textrm{new})} + \beta d\,.
$$

By keeping a component of the current search direction $d$ in the new search direction, we aim to prevent the dreaded zig-zag problem of vanilla gradient descent.  The fully worked details of this step are beyond the scope of this unit (see MXB226!).  But it is possible to show that the optimal formula for $\beta$ is
$$
\beta = \frac{{r^{(\textrm{new})}}^\top H_\gamma d}{d^\top H_\gamma d}\,.
$$

With this formula, the search directions become **conjugate**, giving the method its name, where by conjugate we mean they satisfy:
$$
{d^{(\textrm{new})}}^\top H_\gamma d = 0\,.
$$

Hence, after suitable initialisation, the conjugate gradient method proceeds according to the following classic five-line algorithm:
```
repeat
  α = -(d'*r) / (d'*Hᵧ*d)    # optimal step size
  s ⬅ s + α*d               # update iterate
  r ⬅ r + α*Hᵧ*d            # update gradient
  β = r'*Hᵧ*d / (d'*Hᵧ*d)    # direction weighting
  d ⬅ -r + β*d              # update search direction
end
```

(Almost every line in the algorithm can be written in multiple equivalent ways, so don't be surprised if you see variants of this algorithm in other sources.)

# Hessian-free Newton-CG

So by applying the conjugate gradient method to the local quadratic model, we can overcome the $\mathcal{O}(P^3)$ cost associated with solving the Hessian linear system
$$
H_\gamma \Delta p = -g\,.
$$

Instead, as we can see from the algorithm above, the Hessian matrix is involved in the algorithm only in the form of Hessian-vector products:
$$H_\gamma d = (H + \gamma I) d = Hd + \gamma d\,.$$

What remains is only the $\mathcal{O}(P^2)$ cost of forming the Hessian matrix $H$.

But prepare to be even more amazed, because remarkably we can make *even that cost disappear*!

Because $H$ is not just any matrix -- because it is the Jacobian of the gradient $G$ -- there is a way to compute products $H d$ _without ever forming $H$_.  In fact, it's a one-liner!
$$
H(p) d = \frac{\textrm{d}}{\textrm{d}t} G(p + td)|_{t = 0}\,.
$$

You're asked to verify this formula in the exercises.  We're just making clever use of the chain rule.

What's more, because this is a derivative with respect to the scalar variable $t$, forward mode automatic differentiation can calculate it for the same cost as just evaluating $G$, namely $\mathcal{O}(P)$ work!

Here's the code.

In [ ]:
# Calculates H(p) * d
Hvecprod(p,d) = derivative(t->loss_and_gradient(p+t*d; U, Y)[2], AutoForwardDiff(), 0.0)

#
We can test its validity against the full computation using $H$.

In [ ]:
# Form the full Hessian
H(p) = jacobian(p->loss_and_gradient(p; U, Y)[2], AutoForwardDiff(), p)

# Random direction to try it on
d = randn(length(p0))

# Compare the Hessian-free formula to the full calculation
maximum(abs, Hvecprod(p0,d) - H(p0)*d)

#
But the runtime for `Hvecprod(p,d)` is only $\mathcal{O}(P)$, compared to $\mathcal{O}(P^2)$ for `H(p)`.

In [ ]:
@time Hvecprod(p0, d);
@time H(p0);

#
So with this final trick, we have eliminated any need to ever actually form a Hessian matrix!  We have derived a **Hessian-free Newton-CG method**.  The outer level iteration is Newton applied to the loss function.  The inner level iteration is CG applied fully Hessian-free to the local quadratic model.

We can see it completely crushes the _optimal_ gradient descent in just 30 seconds of runtime.

In [ ]:
# Hessian-free Newton-CG method with regularisation
maxiters = 500
ncgsteps = 5

p = p0
newton_loss_trace = []

@time for i = 1:maxiters

    loss, g = loss_and_gradient(p; U, Y)
    push!(newton_loss_trace, loss)  # record the loss value

    γ = norm(g)   # regularisation parameter

    # Initialisation for conjugate gradient (i.e. at s = 0)
    Δp = zero(g)  # initial value for the Newton step
    r =  g        # initial gradient of local quadratic model
    d = -r        # initial search direction

    # Conjugate gradient iterations
    for k = 1:ncgsteps
        Hᵧd =  Hvecprod(p,d) + γ*d  # H(p + γI) * d
        α = -(d'*r) / (d'*Hᵧd)      # optimal inner step size
        Δp +=  α*d                  # take the inner step
        r +=  α*Hᵧd                 # update the gradient
        β =  r'*Hᵧd / (d'*Hᵧd)      # conjugacy coefficient
        d = -r + β * d              # new descent direction
    end

    # Take the Newton step
    p += Δp

end

lines!(newton_loss_trace/newton_loss_trace[1], label="Newton-CG")
axislegend(ax)
fig

# First-order versus second-order methods
Clearly the takeaway from our investigation is that _incorporating curvature information helps_.  This is the distinguishing feature between **first-order methods** like gradient descent, and **second-order methods** like Newton.

But between the two extremes of vanilla gradient descent and Newton, there lies a spectrum of possibilities.  Some methods can be viewed as starting with gradient descent and injecting some form of curvature information somehow.  We will cover these ideas in a later lesson in the context of **stochastic gradient descent**.

Other methods can be viewed as starting with full Newton and replacing the Hessian linear system solve with something more tractable. We saw the example of the Newton-CG method, which solves the Hessian linear system only approximately (via inner CG iterations).  There are also [quasi-Newton methods](https://en.wikipedia.org/wiki/Quasi-Newton_method) which replace the Hessian entirely with an approximation which is cheaper to form and solve.  The information to build this approximation is extracted from the history of the iterations: the gradients and steps that were computed so far.  The details of how this information is used, and what approximation is built, differ between methods.

# L-BFGS
[Kochenderfer and Wheeler Section 5.2]

For example, the **BFGS** method (named for its creators Broyden-Fletcher-Goldfarb-Shanno) and its "low-memory" variant **L-BFGS** iteratively build an approximation of the _inverse_ of the Hessian matrix.  Initially this approximation may be simply the identity matrix, so early iterations are just gradient descent. But as the iterations proceed, the past history is used to update this estimate, progressively incorporating more curvature information into the approximation.  Crucially, in L-BFGS the matrix is never formed explicitly, and it exists in memory only in the form of a low rank approximation in terms of a small number (10, say) of outer products. The approximation is also carefully designed to ensure it remains positive definite throughout the iterations.

L-BFGS is an excellent optimisation algorithm for many applications in machine learning. Its implementation is rather fiddly however, so we will not build a version ourselves.  Instead we will use a reference implementation from the `Optim` package.

In [ ]:
# L-BFGS algorithm and options
algorithm = Optim.LBFGS()
options   = Optim.Options(iterations = 500, store_trace = true)

# Objective returns only function value and gradient, no Hessian
objective = only_fg(p->loss_and_gradient(p; U,Y))

# Run the optimiser
optresult = Optim.optimize(objective, p0, algorithm, options)

#
It completes its 500 iterations in just seconds.  From the output you can see that only around 1900 evaluations of the loss and gradient were required -- i.e. less than 4 evaluations per iteration.

The convergence of L-BFGS for this problem is rather comparable to Newton-CG.  It is also an exceptionally robust algorithm, so we will not hesitate to use it as our main method in later lessons.  (Whereas our Newton-CG implementation lacks many of the features that would make it an industrial-strength algorithm, capable of handling whatever function we may want to throw at it.)

In [ ]:
lbfgs_loss_trace = Optim.f_trace(optresult)
lines!(lbfgs_loss_trace/lbfgs_loss_trace[1], label="L-BFGS")
axislegend(ax)
fig

# Inference

Now that the model is trained, let's give it a try.  Here's the code to generate new text that we saw in lesson 1.

In [ ]:
# Generate some text with the trained model
function generate_text(model, context, nchars)

    sequence_length = length(context)

    # The result begins with the context
    result = context

    # Convert the start text to character indices
    input = string_to_idxs(context)

    # Generate new characters one by one
    for i = 1:nchars

        # Get the model's probability predictions for the next character
        probs = model(input)[:]

        # Sample the next character based on the probabilities
        next_char_idx = sample(1:vocab_size, Weights(probs))

        # Append the character to the result, which becomes part of the input for the next iteration
        result *= idx_to_char(next_char_idx)
        input = string_to_idxs(result[end-sequence_length+1:end])
    end

    return result

end

# Trained parameters
We need the trained parameters.  We might as well use the L-BFGS result.

In [ ]:
p = Optim.minimizer(optresult)

# Generating text

Let's give it a try!  We define our convenience function for inference, then generate new text using the model.

The text is a little better than random gibberish, but hardly resembles real English text.  Why is this?  Simply, we haven't trained on enough data.  With only 1000 samples it's hardly enough for a model to learn the intricacies of English text.  So what it's learned from its small training set doesn't generalise very well to new text it hasn't seen.

In [ ]:
model(u) = forward(u, p)[1]  # holds the parameter values fixed as p

# Generate 5 text sequences
for i = 1:5
  newtext = generate_text(model, "alice said", 50)
  println(newtext)
end

# Local and global minima

You might be wondering whether any of our optimisation algorithms really stand a chance of finding the _global_ minimum of the loss function.  After all, gradient descent and Newton both converge to _local_ minima.  Once the method has found a local minimum it's happy to just sit there: the gradient is zero, so there is no further progress to be made.

If the loss function has a large number of local minima with nonetheless high values of the loss, then we may expect that most of the time our methods will converge to one of those.  That would be bad -- the loss is still very large, but there is no way to make further progress.  This was indeed a concern that researchers had in the early days of neural network-based AI research.

The reality in high dimensions turns out to be different though.  A good reference is [Choromanska et. al (2015)](https://arxiv.org/pdf/1412.0233).  We repeat two of their main points below:

* For large-size networks, most local minima are equivalent and yield similar performance on a test set.

* The probability of finding a "bad" (high value) local minimum is non-zero for small-size networks but decreases quickly with network size.

There is some theory to support these observations in simplified cases, but still no fully developed theory that covers loss functions for neural networks in general.

But the observations are repeatedly borne out in experiments on real networks.  Yes, there are local minima, including local minima that are "bad".  But the bad local minima are exponentially outnumbered by _saddle points_.  And saddle points do *not* stall (regularised) Newton or gradient descent.

Why?  Because unless you happen to land _exactly_ on a saddle point (which happens with probability zero), there are still directions that lead downhill in the vicinity of the saddle point.  That's _why_ it's a saddle point: some directions lead up, some directions lead down.  Gradient descent and Newton will follow a downhill direction.

In terms of the Hessian matrix, a saddle point is associated with some of the eigenvalues being positive, and some being negative.  In high dimensions, this mix of positive and negative eigenvalues (saddle) is overwhelmingly more likely than all the eigenvalues being positive (local minimum).  So saddle points are everywhere, but they're not a concern.

So what of the local minima that do exist -- after all, the iteration will ultimately have to converge to one of them. Again, in high dimensional networks, the observation (partly supported by theory) is that these tend to cluster around the true global minimum.  So any local minimum that you converge to will be of the "good" variety -- it will yield a model that generalises about as well as any other, and comparable to the globally optimal model.

# Conclusions

In this lesson we learned:

* how to reliably initialise the weight matrices in a neural network before training
* how a line search turns the multivariable loss into a one-variable function $\phi(\alpha)$
* why gradient descent tends to zig-zag in narrow valleys and can therefore converge slowly
* how conjugate gradient can approximately solve the Newton system without explicitly inverting the Hessian
* how Hessian-vector products can be computed efficiently with automatic differentiation, making Newton's method practical in principle
* about the quasi-Newton method L-BFGS, and how to use a reference implementation to train our model

That concludes the presentation of our character-level language model (CLLM).  We have built it from the ground up, and finally now have it training successfully.

In the next lesson we will explore the structure of a large language model (LLM), and in particular introduce the transformer architecture.